<a href="https://colab.research.google.com/github/KenPham0604/US_Accident_Severity_Prediction/blob/main/RQ1_LightGBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1 — Setup

from google.colab import drive
import pandas as pd
import numpy as np
import zipfile, shutil, os, gc

pd.set_option("display.max_columns", None)
SEED = 42
np.random.seed(SEED)

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/ADY201m/'

def optimize_ram(df, verbose=True):
    ram_before = df.memory_usage(deep=True).sum() / (1024**2)
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_integer_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        elif pd.api.types.is_float_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="float")
        elif col_type == "object":
            if df[col].nunique() / len(df) < 0.5:
                df[col] = df[col].astype("category")
    ram_after = df.memory_usage(deep=True).sum() / (1024**2)
    if verbose:
        print(f"RAM: {ram_before:.2f} MB -> {ram_after:.2f} MB")
    return df

def read_zip_chunking(file_name, directory=DRIVE_DIR, chunksize=500_000):
    drive_zip = f'{directory}{file_name}.zip'
    local_zip = f'/content/{file_name}.zip'
    shutil.copy(drive_zip, local_zip)
    chunks = []
    with zipfile.ZipFile(local_zip, 'r') as z:
        csv_name = z.namelist()[0]
        with z.open(csv_name) as f:
            for i, chunk in enumerate(pd.read_csv(f, chunksize=chunksize)):
                chunks.append(chunk)
                print(f"  Chunk {i+1}: {sum(len(c) for c in chunks):,} rows")
    df = pd.concat(chunks, ignore_index=True)
    del chunks; gc.collect()
    os.remove(local_zip)
    return df

print("✅ Setup completed")

In [ ]:
# 2 — Read cleaned file (34 columns, without Year/Month/Season)

df = read_zip_chunking('US_Accidents_cleaned_remove_distance')
print(f"\nShape: {df.shape}")
df = optimize_ram(df)
gc.collect()
print(df.dtypes)
print(df.columns.tolist())

In [ ]:
# 3 — Extract Year/Month/Hour/DayOfWeek/Season from Start_Time
df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce")

num_nat = df["Start_Time"].isna().sum()
print(f"Number of NaT rows in Start_Time: {num_nat:,}")
if num_nat > 0:
    df = df.dropna(subset=["Start_Time"])

df["Year"] = df["Start_Time"].dt.year.astype("int16")
df["Month"] = df["Start_Time"].dt.month.astype("int8")
df["Hour"] = df["Start_Time"].dt.hour.astype("int8")
df["DayOfWeek"] = df["Start_Time"].dt.dayofweek.astype("int8")

season_map = {12:"Winter",1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",
              6:"Summer",7:"Summer",8:"Summer",9:"Fall",10:"Fall",11:"Fall"}
df["Season"] = df["Month"].map(season_map).astype("category")

df = df.drop(columns=["Start_Time", "End_Time"])  # End_Time dropped due to data leakage

print(f"Shape after time extraction: {df.shape}")
gc.collect()

In [ ]:
# 4 — Create Severity_High (1 if Severity>=3, otherwise 0)
# according to the definition "Severity 3-4 is considered high severity"
# Group features: Weather, Time, Light, Infrastructure

df["Severity_High"] = (df["Severity"] >= 3).astype("int8")

print("Original Severity distribution:")
print((df["Severity"].value_counts(normalize=True).sort_index().round(4) * 100))
print("\nSeverity_High distribution (analysis target variable):")
print((df["Severity_High"].value_counts(normalize=True).round(4) * 100))

group_geographic = ["Start_Lat", "Start_Lng", "Street", "City", "County", "State", "Zipcode", "Timezone", "Airport_Code"]
group_weather = ["Temperature(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)",
                 "Wind_Direction", "Wind_Speed(mph)", "Weather_Condition"]
group_time = ["Year", "Month", "Hour", "DayOfWeek", "Season"]
group_light = ["Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]
group_infra = ["Amenity", "Crossing", "Give_Way", "Junction", "No_Exit",
              "Railway", "Station", "Stop", "Traffic_Signal"]

all_features = group_weather + group_time + group_light + group_infra + group_geographic
print(f"\nTotal features used for RQ1: {len(all_features)}")
print(df.columns.tolist())

In [ ]:
# 6 — Split Train/Test, train LightGBM (fast, handles categories well)

!pip install lightgbm shap -q
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier

X = df[all_features]
y = df["Severity_High"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

# Handle imbalance using class weights with a ceiling cap (based on prior
# empirical findings: cap ~2.5-3 yields better Macro F1 than default "balanced")
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train)
natural_weights = compute_class_weight("balanced", classes=classes, y=y_train)
WEIGHT_CAP = 2.5
capped_weights = np.minimum(natural_weights, WEIGHT_CAP)
class_weight_dict = dict(zip(classes, capped_weights))
print(f"Class weight (capped at={WEIGHT_CAP}): {class_weight_dict}")

model = LGBMClassifier(
    n_estimators=250,
    num_leaves=31,
    class_weight=class_weight_dict,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)

print("Training LightGBM for RQ1 analysis...")
model.fit(X_train, y_train)

from sklearn.metrics import accuracy_score, f1_score
y_pred = model.predict(X_test)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro F1: {f1_score(y_test, y_pred, average='macro'):.4f}")

gc.collect()

In [ ]:
# 7 — Feature Importance (Gain-based) from LightGBM

feature_importance = pd.DataFrame({
    "Feature": all_features,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

# Assign group to each feature to aggregate by Weather/Time/Light/Infrastructure
def assign_group(feat):
    if feat in group_weather: return "Weather"
    if feat in group_time: return "Time"
    if feat in group_light: return "Light"
    if feat in group_infra: return "Infrastructure"
    if feat in group_geographic: return "Geographic"
    return "Other"

feature_importance["Group"] = feature_importance["Feature"].apply(assign_group)

print("=== Feature Importance (Individual variables) ===")
display(feature_importance)

print("\n=== Total Importance by GROUP (Weather/Time/Light/Infrastructure/Geographic) ===")
group_summary = feature_importance.groupby("Group")["Importance"].sum().sort_values(ascending=False)
display(group_summary)

feature_importance.to_csv('/content/drive/MyDrive/ADY201m/RQ1_feature_importance.csv', index=False)

In [ ]:
# 8 — SHAP ANALYSIS (CORRECTED)
# Sample: 200,000 rows
# Model: LightGBM Binary Classifier
# Target: Severity_High = 1 (Severity 3 and 4)

import shap
import numpy as np
import pandas as pd
import gc

# 1. Sampling

SHAP_SAMPLE = min(200_000, len(X_test))

X_shap_sample = X_test.sample(
    n=SHAP_SAMPLE,
    random_state=SEED
)

# Always use the actual model feature order
if hasattr(model, "feature_name_"):

    model_features = list(model.feature_name_)

    if (
        len(model_features) == X_shap_sample.shape[1]
        and set(model_features) == set(X_shap_sample.columns)
    ):
        X_shap_sample = X_shap_sample[model_features]
    else:
        raise ValueError(
            "Model feature names do not match X_test. "
            "Check preprocessing and feature order."
        )

feature_names = X_shap_sample.columns.tolist()

print(f"SHAP sample size: {len(X_shap_sample):,}")
print(f"Number of features: {len(feature_names)}")
print("Explained class: Severity_High = 1 (Severity 3–4)")
print("Model: LightGBM Binary Classifier")

# 2. Calculate SHAP

print("Calculating SHAP values...")

explainer = shap.TreeExplainer(model)

shap_values = explainer.shap_values(
    X_shap_sample
)

# 3. Select class 1

classes = np.asarray(model.classes_)

if 1 not in classes:
    raise ValueError("Class 1 not found in model.classes_")

class1_idx = int(np.where(classes == 1)[0][0])

if isinstance(shap_values, list):

    shap_values_class1 = np.asarray(
        shap_values[class1_idx]
    )

else:

    shap_values = np.asarray(shap_values)

    if shap_values.ndim == 3:

        if shap_values.shape[1] == len(feature_names):
            shap_values_class1 = shap_values[
                :, :, class1_idx
            ]

        elif shap_values.shape[2] == len(feature_names):
            shap_values_class1 = shap_values[
                :, class1_idx, :
            ]

        else:
            raise ValueError(
                f"Unexpected SHAP shape: {shap_values.shape}"
            )

    elif shap_values.ndim == 2:

        shap_values_class1 = shap_values

    else:

        raise ValueError(
            f"Unexpected SHAP shape: {shap_values.shape}"
        )

assert shap_values_class1.shape == X_shap_sample.shape

# 4. Mean Absolute SHAP by feature

mean_abs_shap = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Abs_SHAP": (
        np.abs(shap_values_class1).mean(axis=0)
    )
})

mean_abs_shap = mean_abs_shap.sort_values(
    "Mean_Abs_SHAP",
    ascending=False
).reset_index(drop=True)

mean_abs_shap["Rank"] = np.arange(
    1, len(mean_abs_shap) + 1
)

mean_abs_shap["Group"] = (
    mean_abs_shap["Feature"].apply(assign_group)
)

# Check all features have a defined group
if mean_abs_shap["Group"].isna().any():

    missing_features = mean_abs_shap.loc[
        mean_abs_shap["Group"].isna(),
        "Feature"
    ].tolist()

    raise ValueError(
        f"Features without factor groups: {missing_features}"
    )

# 5. SHAP contribution by group

# Method 1: Original Total Mean Absolute SHAP
# Method 2: Average Mean Absolute SHAP per Feature

tong_hop_shap_nhom = (
    mean_abs_shap
    .groupby("Group")["Mean_Abs_SHAP"]
    .agg(["sum", "mean", "count"])
    .rename(columns={
        "sum": "Total_SHAP",
        "mean": "Average_SHAP_Per_Feature",
        "count": "N_Features"
    })
    .sort_values("Total_SHAP", ascending=False)
)

total_shap = tong_hop_shap_nhom["Total_SHAP"].sum()

# Preserve the original contribution percentage
tong_hop_shap_nhom["SHAP_Share_%"] = (
    tong_hop_shap_nhom["Total_SHAP"]
    / total_shap * 100
)

# Method 3: Grouped Net Mean Absolute SHAP
# For each sample:
# 1. Sum signed SHAP values within each factor group
# 2. Take the absolute value of the group sum
# 3. Average across all samples
# This addresses group aggregation concerns.

feature_groups = pd.Series(
    feature_names
).apply(assign_group).to_numpy()

grouped_net_shap = {}

for group in tong_hop_shap_nhom.index:

    # Indices of features belonging to this group
    group_indices = np.where(
        feature_groups == group
    )[0]

    # Sum signed SHAP values per sample
    group_shap_per_sample = (
        shap_values_class1[:, group_indices]
        .sum(axis=1)
    )

    # Mean absolute grouped SHAP
    grouped_net_shap[group] = float(
        np.abs(group_shap_per_sample).mean()
    )

# Add the new metric
tong_hop_shap_nhom["Grouped_Net_MeanAbs"] = (
    pd.Series(grouped_net_shap)
)

# Percentage shares for the new metrics

total_average_shap = (
    tong_hop_shap_nhom["Average_SHAP_Per_Feature"]
    .sum()
)

tong_hop_shap_nhom["Average_SHAP_Share_%"] = (
    tong_hop_shap_nhom["Average_SHAP_Per_Feature"]
    / total_average_shap * 100
)

total_grouped_net_shap = (
    tong_hop_shap_nhom["Grouped_Net_MeanAbs"]
    .sum()
)

tong_hop_shap_nhom["Grouped_Net_Share_%"] = (
    tong_hop_shap_nhom["Grouped_Net_MeanAbs"]
    / total_grouped_net_shap * 100
)

# 5.1. Compare group rankings

group_rank_comparison = pd.DataFrame({
    "Total_SHAP_Rank": (
        tong_hop_shap_nhom["Total_SHAP"]
        .rank(ascending=False, method="min")
        .astype(int)
    ),

    "Average_Per_Feature_Rank": (
        tong_hop_shap_nhom["Average_SHAP_Per_Feature"]
        .rank(ascending=False, method="min")
        .astype(int)
    ),

    "Grouped_Net_Rank": (
        tong_hop_shap_nhom["Grouped_Net_MeanAbs"]
        .rank(ascending=False, method="min")
        .astype(int)
    )
})

group_rank_comparison = (
    group_rank_comparison
    .sort_values("Grouped_Net_Rank")
)

# 6. Output

print("\n=== FEATURE SHAP RANKING ===")

display(
    mean_abs_shap[
        ["Rank", "Feature", "Mean_Abs_SHAP", "Group"]
    ]
)

print("\n=== SHAP BY FACTOR GROUP ===")

display(
    tong_hop_shap_nhom.round(6)
)

print("\n=== GROUP AGGREGATION RANK COMPARISON ===")

display(group_rank_comparison)

print("\n=== TOP FEATURE ===")

print(
    mean_abs_shap.iloc[0]["Feature"],
    "—",
    mean_abs_shap.iloc[0]["Mean_Abs_SHAP"]
)

print("\n=== TOP FACTOR GROUP (GROUPED NET SHAP) ===")

top_group = (
    tong_hop_shap_nhom["Grouped_Net_MeanAbs"]
    .idxmax()
)

print(
    top_group,
    "—",
    tong_hop_shap_nhom.loc[
        top_group, "Grouped_Net_MeanAbs"
    ]
)

# 7. Save results

output_path = "/content/drive/MyDrive/ADY201m/"

mean_abs_shap.to_csv(
    output_path + "RQ1_mean_abs_shap.csv",
    index=False
)

tong_hop_shap_nhom.to_csv(
    output_path + "RQ1_group_shap.csv"
)

# New: save group aggregation comparison
group_rank_comparison.to_csv(
    output_path + "RQ1_group_SHAP_rank_comparison.csv"
)

print("\n✅ SHAP analysis completed.")

gc.collect()

In [ ]:
# 9 — Visualization of RQ1 Results
# Publication-ready figures

import matplotlib.pyplot as plt
import shap

# Global plotting style

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": 300
})

# Figure 1: SHAP Summary Plot

shap.summary_plot(
    shap_values_class1,
    X_shap_sample,
    feature_names=feature_names,
    cmap=plt.cm.Blues,    # blue-gray consistent throughout the paper
    show=False
)

plt.tight_layout()

plt.savefig(
    "/content/drive/MyDrive/ADY201m/RQ1_shap_summary.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Figure 2: SHAP Contribution by Factor Group
# Grouped Net Mean Absolute SHAP
# Color palette based on the paper guideline
# Explicit color mapping for all five factor groups

group_colors = {
    "Weather": "#D5E0E0",
    "Time": "#BFD0D7",
    "Light": "#E0E7EB",
    "Infrastructure": "#567A89",
    "Geographic": "#8FAEBF"
}

# Select only the metric used for the final figure

group_plot = (
    tong_hop_shap_nhom["Grouped_Net_MeanAbs"]
    .sort_values(ascending=True)
)

group_shares = (
    tong_hop_shap_nhom["Grouped_Net_Share_%"]
    .reindex(group_plot.index)
)

bar_colors = [
    group_colors.get(group, "#A0A0A0")
    for group in group_plot.index
]

# Visualization

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.barh(
    group_plot.index,
    group_plot.values,
    color=bar_colors,
    edgecolor="black",
    linewidth=0.7
)

ax.set_xlabel(
    "Grouped Mean Absolute SHAP Value",
    fontsize=10
)

ax.set_ylabel(
    "Factor Group",
    fontsize=10
)

ax.set_title(
    "Grouped SHAP Importance for High-Severity Prediction",
    fontsize=11,
    pad=10
)

# Numerical labels and shares

max_value = group_plot.max()

for bar, share in zip(
    bars,
    group_shares.values
):

    value = bar.get_width()

    ax.text(
        value + max_value * 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f} ({share:.1f}%)",
        va="center",
        fontsize=9
    )

ax.set_xlim(
    0,
    max_value * 1.35
)

# Academic-style grid
ax.grid(
    axis="x",
    linestyle="--",
    linewidth=0.5,
    alpha=0.5
)

ax.set_axisbelow(True)

# Remove unnecessary borders
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.savefig(
    "/content/drive/MyDrive/ADY201m/RQ1_nhom_yeu_to.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\n✅ RQ1 visualization completed and saved to Google Drive.")